In [1]:
! pip install pandas
! pip install tensorflow

In [2]:
! pip install nlpaug

In [3]:
! pip install nltk scikit-learn

In [4]:
! pip install tensorflow-datasets

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding, TextVectorization, Bidirectional
import tensorflow_datasets as tfds
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

2025-07-09 21:44:30.269419: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-09 21:44:30.345758: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-09 21:44:30.388045: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752076770.441529    1319 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752076770.456154    1319 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752076770.559440    1319 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [6]:
dataset = tfds.load('imdb_reviews', as_supervised=True)

E0000 00:00:1752076776.124080    1319 cuda_executor.cc:1228] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1752076776.128856    1319 gpu_device.cc:2341] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [7]:
train_dataset, test_dataset = dataset['train'], dataset['test']

batch_size = 32

train_dataset = train_dataset.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [8]:
vectorizer = TextVectorization(max_tokens=10000, output_sequence_length=500)
vectorizer.adapt(train_dataset.map(lambda x, y: x))

2025-07-09 21:44:36.288113: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2025-07-09 21:44:38.765121: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [9]:
model = Sequential()
model.add(vectorizer)
model.add(Embedding(input_dim=len(vectorizer.get_vocabulary()), output_dim=64, mask_zero=True))
model.add(Bidirectional(LSTM(64, return_sequences=True))),
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(32))),
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [12]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [13]:
model.fit(train_dataset, epochs=10, validation_data=test_dataset, callbacks=[early_stop])

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 217s 272ms/step - accuracy: 0.5718 - loss: 0.6593 - val_accuracy: 0.8496 - val_loss: 0.3574
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 212s 271ms/step - accuracy: 0.8792 - loss: 0.3051 - val_accuracy: 0.8781 - val_loss: 0.2923
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 234s 300ms/step - accuracy: 0.9230 - loss: 0.2135 - val_accuracy: 0.8771 - val_loss: 0.2924
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 235s 300ms/step - accuracy: 0.9405 - loss: 0.1742 - val_accuracy: 0.8727 - val_loss: 0.3337
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 204s 260ms/step - accuracy: 0.9533 - loss: 0.1410 - val_accuracy: 0.8698 - val_loss: 0.3583


In [14]:
text = "I hate this movie!"
text = tf.constant([text]) 
print("Shape:", text.shape)

Shape: (1,)


In [15]:
prediction = model.predict(text)

label_text = "positive" if prediction[0][0] > 0.5 else "negative"
print("Predicted sentiment:", label_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 429ms/step
Predicted sentiment: negative


In [24]:
model.save('sentiment_model.keras')